# 1. Triage & Alert Correlation

Welcome to the SOC. A real analyst's day doesn't start with "hunt for evil" — it starts with **a queue of alerts**, and the first skill you need is deciding **what to look at first**.

This notebook teaches:

1. How raw alerts get grouped into **incidents** by shared entities (the "correlate" step).
2. How to **triage** an incident queue: not just severity, but recency, confidence, and blast radius.
3. A bad → best progression you can use on the exam AND in a real SOC.

> **SC-200 mapping**: "Triage security alerts and incidents" and "Manage incidents in Microsoft Defender XDR and Microsoft Sentinel".


## Setup

This lab reuses the mini-SIEM from Lab 1. Make sure it's running:

```bash
cd ../../01-build-a-siem && docker compose up -d
```

Then, in VS Code:
1. Pick the **`.venv` kernel** from this folder (top-right kernel picker).
2. If it's missing, reload the window (`Cmd+Shift+P` → `Reload Window`).

All cells talk to `http://localhost:8000` — the same mini-SIEM container seeded with a realistic multi-stage attack plus normal background traffic.


In [1]:
import httpx
from collections import Counter, defaultdict
from datetime import datetime

SIEM = 'http://localhost:8000'

# Sanity check: can we reach the SIEM?
health = httpx.get(f'{SIEM}/health').json()
print('SIEM health:', health)

dashboard = httpx.get(f'{SIEM}/dashboard').json()
print('Dashboard:', dashboard)


SIEM health: {'status': 'ok', 'service': 'mini-siem', 'counts': {'logs': 172, 'analytics_rules': 6, 'alerts': 6, 'incidents': 6, 'playbooks': 5}}
Dashboard: {'total_logs': 172, 'tables': ['AzureFirewall', 'DeviceEvents', 'EmailEvents', 'SigninLogs'], 'active_rules': 6, 'open_alerts': 0, 'open_incidents': 0, 'severity_breakdown': {'High': 4, 'Medium': 2}}


## Step 1 — Look at raw alerts

Each **alert** is a single detection firing once. The mini-SIEM exposes them at `GET /alerts`.

Think of alerts as tickets an analyst could be asked to read. There are usually dozens per hour in a real tenant.


In [2]:
alerts = httpx.get(f'{SIEM}/alerts').json()
print(f'Total alerts: {len(alerts)}\n')

for a in alerts[:10]:
    print(f"[{a['severity']:<8}] {a['rule_name']:<32} tactic={a['tactic']!s:<18} status={a['status']}")
    print(f"           {a['title']}")


Total alerts: 6

[High    ] Brute force sign-in              tactic=CredentialAccess   status=InIncident
           Brute force sign-in: alice@contoso.com (15 events)
[Medium  ] Sign-in from suspicious location tactic=InitialAccess      status=InIncident
           Sign-in from suspicious location (7 events)
[High    ] Suspicious process execution     tactic=Execution          status=InIncident
           Suspicious process execution (3 events)
[High    ] Lateral movement detected        tactic=LateralMovement    status=InIncident
           Lateral movement detected: alice (3 events)
[Medium  ] Phishing email delivered         tactic=InitialAccess      status=InIncident
           Phishing email delivered (1 events)
[High    ] Outbound traffic to known malicious IP tactic=Exfiltration       status=InIncident
           Outbound traffic to known malicious IP (4 events)


## Step 2 — Correlate alerts into incidents

Several alerts can be **the same story**: a brute-force alert and a "sign-in from suspicious location" alert for the same user are almost certainly one incident, not two.

In Defender XDR and Sentinel, the platform groups them automatically. Our mini-SIEM exposes the same idea at `POST /incidents/correlate` — it groups "New" alerts by shared entities (e.g. `UserPrincipalName`).

The call is **idempotent-ish**: once an alert's status moves to `InIncident`, re-running won't create a duplicate incident for it.


In [3]:
correlated = httpx.post(f'{SIEM}/incidents/correlate').json()
print('Correlation result:', correlated)

incidents = httpx.get(f'{SIEM}/incidents').json()
print(f'\nTotal incidents now: {len(incidents)}')
for inc in incidents:
    print(f"  {inc['id']}  sev={inc['severity']:<8} status={inc['status']:<6}  {inc['title']}")


Correlation result: {'incidents_created': []}

Total incidents now: 6
  INC-ddb928  sev=High     status=Closed  Incident: Brute force sign-in (UserPrincipalName=alice@contoso.com)
  INC-6dfb14  sev=Medium   status=Closed  Incident: Sign-in from suspicious location (multiple)
  INC-97f9ac  sev=High     status=Closed  Incident: Suspicious process execution (multiple)
  INC-474aee  sev=High     status=Closed  Incident: Lateral movement detected (AccountName=alice)
  INC-24ad2d  sev=Medium   status=Closed  Incident: Phishing email delivered (multiple)
  INC-1c3741  sev=High     status=Closed  Incident: Outbound traffic to known malicious IP (multiple)


## Step 3 — Bad triage: first-in-first-out

The **bad** approach is to just walk the queue top to bottom. You will burn hours on low-impact alerts while a real breach sits unanswered.


In [4]:
print('❌ BAD: first-in-first-out (no prioritization)')
for inc in incidents:
    print(f"  → work on {inc['id']}  ({inc['severity']})")


❌ BAD: first-in-first-out (no prioritization)
  → work on INC-ddb928  (High)
  → work on INC-6dfb14  (Medium)
  → work on INC-97f9ac  (High)
  → work on INC-474aee  (High)
  → work on INC-24ad2d  (Medium)
  → work on INC-1c3741  (High)


## Step 4 — Better triage: severity + recency

A small step up is to sort by severity, tie-break by recency. This is what most junior SOCs actually do.


In [5]:
SEV_ORDER = {'Critical': 4, 'High': 3, 'Medium': 2, 'Low': 1, 'Informational': 0}

better = sorted(
    incidents,
    key=lambda i: (SEV_ORDER.get(i['severity'], 0), i['created_at']),
    reverse=True,
)
print('⚠️  BETTER: severity then recency')
for inc in better:
    print(f"  {inc['severity']:<8} {inc['created_at']}  {inc['id']}  {inc['title']}")


⚠️  BETTER: severity then recency
  High     2026-04-20 21:57:22  INC-ddb928  Incident: Brute force sign-in (UserPrincipalName=alice@contoso.com)
  High     2026-04-20 21:57:22  INC-97f9ac  Incident: Suspicious process execution (multiple)
  High     2026-04-20 21:57:22  INC-474aee  Incident: Lateral movement detected (AccountName=alice)
  High     2026-04-20 21:57:22  INC-1c3741  Incident: Outbound traffic to known malicious IP (multiple)
  Medium   2026-04-20 21:57:22  INC-6dfb14  Incident: Sign-in from suspicious location (multiple)
  Medium   2026-04-20 21:57:22  INC-24ad2d  Incident: Phishing email delivered (multiple)


## Step 5 — Best triage: severity + confidence + blast radius + containment

Real SOC triage weighs four things, not one:

| Factor | Question |
|---|---|
| **Severity** | How bad is it if this is real? |
| **Confidence** | How likely is the detection real vs a false positive? Multiple correlated alerts ⇒ higher confidence. |
| **Blast radius** | How many users/devices/services are affected? |
| **Containment state** | Has automation already contained the threat? If yes, drop the urgency. |

We'll approximate each factor from the incident data and produce a **priority score**. Higher = look at first.


In [6]:
# Pull each incident with its alerts (so we can count signal and entities)
def score(incident):
    detail = httpx.get(f"{SIEM}/incidents/{incident['id']}").json()
    alerts = detail.get('alerts', [])

    sev = SEV_ORDER.get(incident['severity'], 0)

    # Confidence: how many independent alerts fired for this story?
    confidence = min(len(alerts), 5)  # cap so one noisy rule doesn't dominate

    # Blast radius: unique devices + users mentioned in evidence
    actors = set()
    for a in alerts:
        import json as _json
        for ev in _json.loads(a['evidence'] or '[]'):
            for key in ('UserPrincipalName', 'DeviceName', 'AccountName', 'SourceIP'):
                if ev.get(key):
                    actors.add((key, ev[key]))
    blast = min(len(actors), 10)

    # Containment: new = not contained yet (add urgency), closed = contained
    containment_penalty = 0 if incident['status'] == 'New' else -2

    # Weighted priority score
    priority = sev * 10 + confidence * 3 + blast * 2 + containment_penalty
    return priority, {
        'severity': incident['severity'],
        'alerts': len(alerts),
        'blast': len(actors),
        'status': incident['status'],
        'priority': priority,
    }

ranked = [(score(i), i) for i in incidents]
ranked.sort(key=lambda x: x[0][0], reverse=True)

print('✅ BEST: severity + confidence + blast radius + containment')
print(f'{"incident":<12} {"prio":>4}  sev   alerts  blast  status   title')
for (p, meta), inc in ranked:
    print(f"{inc['id']:<12} {meta['priority']:>4}  {meta['severity']:<5} {meta['alerts']:>6}  {meta['blast']:>5}  {meta['status']:<7}  {inc['title']}")


✅ BEST: severity + confidence + blast radius + containment
incident     prio  sev   alerts  blast  status   title
INC-97f9ac     39  High       1      4  Closed   Incident: Suspicious process execution (multiple)
INC-474aee     35  High       1      2  Closed   Incident: Lateral movement detected (AccountName=alice)
INC-ddb928     33  High       1      1  Closed   Incident: Brute force sign-in (UserPrincipalName=alice@contoso.com)
INC-1c3741     33  High       1      1  Closed   Incident: Outbound traffic to known malicious IP (multiple)
INC-6dfb14     23  Medium      1      1  Closed   Incident: Sign-in from suspicious location (multiple)
INC-24ad2d     21  Medium      1      0  Closed   Incident: Phishing email delivered (multiple)


## What you just did (SC-200 mapping)

| You did... | Real portal equivalent |
|---|---|
| `GET /alerts` | Defender XDR → Alerts queue |
| `POST /incidents/correlate` | XDR's automatic alert-to-incident correlation |
| Ranked by severity + confidence + blast + containment | The **Priority** column and the analyst's mental model |
| Inspected incident with its alerts | Incident page → Alerts tab |

### Exam tips

- **Correlation is the point of an incident.** Don't work alerts one by one when they share entities.
- **Severity alone is a weak ranking.** Two High incidents with different blast radius are not equal.
- If **automatic attack disruption** already contained the threat, triage urgency drops — but investigation urgency does not.

➡️ Next: [02 — Entity-centric investigation](02_entity_pivot_investigation.ipynb)
